# DEH 30-Day PySpark Challenge
## Day 5 — Selecting Columns

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Complete the assignment cells at the bottom

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

## Step 4 — Load orders with explicit schema

In [ ]:
orders_schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("order_date",     DateType(),    True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount_pct",   IntegerType(), True),
    StructField("status",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("region",         StringType(),  True)
])

orders_df = spark.read \
    .option("header", "true") \
    .schema(orders_schema) \
    .csv(f"s3a://{BUCKET}/data/orders.csv")

print(f"Columns: {orders_df.columns}")
orders_df.show(3)

## Step 5 — Three ways to select columns

In [ ]:
# Style 1 — string column names
orders_df.select("order_id", "customer_id", "unit_price").show(3)

In [ ]:
# Style 2 — col() from functions
orders_df.select(
    F.col("order_id"),
    F.col("customer_id"),
    F.col("unit_price")
).show(3)

In [ ]:
# Style 3 — DataFrame subscript notation
orders_df.select(
    orders_df["order_id"],
    orders_df["customer_id"],
    orders_df["unit_price"]
).show(3)

## Step 6 — Aliasing and computing columns

In [ ]:
# Rename columns using alias()
orders_df.select(
    F.col("order_id").alias("id"),
    F.col("unit_price").alias("price"),
    F.col("customer_id").alias("cust_id")
).show(3)

In [ ]:
# Compute new columns in select()
orders_df.select(
    F.col("order_id"),
    F.col("unit_price"),
    F.col("quantity"),
    F.col("discount_pct"),
    (F.col("unit_price") * F.col("quantity")).alias("gross_amount"),
    (F.col("unit_price") * (1 - F.col("discount_pct") / 100)).alias("discounted_price")
).show(3)

## Step 7 — selectExpr()

In [ ]:
# selectExpr — SQL expressions as strings
orders_df.selectExpr(
    "order_id",
    "customer_id",
    "unit_price",
    "unit_price * quantity AS gross_amount",
    "unit_price * (1 - discount_pct / 100) AS discounted_price",
    "UPPER(status) AS status_upper"
).show(3)

In [ ]:
# selectExpr for quick renaming
orders_df.selectExpr(
    "order_id AS id",
    "customer_id AS cust_id",
    "unit_price AS price",
    "order_date AS date"
).show(3)

## Step 8 — Select all except specific columns

In [ ]:
# See all column names
print(orders_df.columns)

# Select all except specific columns
cols_to_exclude = ["discount_pct", "payment_method"]
selected_cols = [c for c in orders_df.columns if c not in cols_to_exclude]
orders_df.select(selected_cols).show(3)

---
## Assignment — Day 5

---

### Task 1
From `orders.csv`, select `order_id`, `customer_id`, `order_date`, and `status` using all three styles — string names, `col()`, and `df["col"]`.  
Confirm all three produce the same result.

In [ ]:
# Task 1 — Write your code here


### Task 2
Using `select()` with `col()`, compute:
- `revenue` = `unit_price * quantity`
- `final_price` = `unit_price * (1 - discount_pct / 100)`

Show `order_id`, `unit_price`, `quantity`, `discount_pct`, `revenue`, and `final_price`.

In [ ]:
# Task 2 — Write your code here


### Task 3
Rewrite Task 2 using `selectExpr()`.  
Add one more expression: `UPPER(region) AS region_upper`.  
Compare the output with Task 2.

In [ ]:
# Task 3 — Write your code here


### Task 4
From `customers.csv`, select all columns except `email` and `country` using the list comprehension approach.  
How many columns does the result have?

In [ ]:
# Task 4 — Write your code here


---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*